# BITS Pilani — Deep Reinforcement Learning — Lab Assignment 1
## Part #1: Adaptive Treatment Recommendation using Multi-Armed Bandits
**Group Number:** 151  
**Submission Deadline:** 8th June, 2026

In [ ]:
# ── Cell 1: Imports, Constants, and Random Seed ──────────────────────────────
# We define the group number as a single constant. Every derived parameter
# (number of medicines, success probabilities, dataset seed) flows from G.

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ─── Group constant (change only this to adapt for a different group) ───
G = 151

# ─── Set seeds for reproducibility ───
# Using the group number as seed ensures that every run produces identical
# results, which is required for evaluation and grading.
random.seed(G)
np.random.seed(G)

print(f"Group Number (G): {G}")
print(f"Random seeds set to {G} for both random and numpy.")

In [ ]:
# ── Cell 2: Compute Number of Medicines (K) and Hidden Probabilities ─────────
# K depends on the group number. Each medicine i has a hidden probability P_i
# that determines how likely a patient is to recover when given that medicine.

# Number of medicines available for this group
K = (G % 3) + 5

# Hidden success probability for each medicine i (0 to K-1)
# Formula: P_i = 0.4 + ((G + i) mod 6) * 0.07
success_probabilities = []
for i in range(K):
    p_i = 0.4 + ((G + i) % 6) * 0.07
    success_probabilities.append(round(p_i, 4))

# Display the computed parameters
print(f"{'='*60}")
print(f"  TASK 1: Dataset Design — Group {G}")
print(f"{'='*60}")
print(f"  Total Medicines (K): {K}")
print(f"\n  Hidden Success Probabilities:")
print(f"  {'Medicine':<12} {'Probability':<12}")
print(f"  {'-'*24}")
for i, p in enumerate(success_probabilities):
    marker = " <-- BEST" if p == max(success_probabilities) else ""
    print(f"  Medicine {i:<3} {p:<12}{marker}")

best_medicine = np.argmax(success_probabilities)
print(f"\n  Best medicine: Medicine {best_medicine} (P = {max(success_probabilities)})")

In [ ]:
# ── Cell 3: Generate Base Patient DataFrame ──────────────────────────────────
# We create a template dataframe with 1000 patients. Each patient has a fixed
# severity score based on their ID. The treatment-related columns (medicine,
# outcome, utility) are left empty — each strategy will fill its own copy.

NUM_PATIENTS = 1000

# Build the base dataframe
base_df = pd.DataFrame({
    "patient_id": range(NUM_PATIENTS),
    "severity_score": [(pid % 5) + 1 for pid in range(NUM_PATIENTS)]
})

# Add empty columns that each strategy will populate on its own copy
base_df["assigned_medicine"] = np.nan
base_df["clinical_outcome"] = np.nan
base_df["utility_score"] = np.nan

print(f"Base dataset created: {len(base_df)} patients")
print(f"Severity distribution: {base_df['severity_score'].value_counts().sort_index().to_dict()}")
print(f"\nFirst 10 rows of the base dataset:")
base_df.head(10)

In [ ]:
# ── Cell 4: Simulate Outcome Helper Function ─────────────────────────────────
# This function simulates what happens when a specific medicine is given to a
# patient. It uses the medicine's hidden success probability to draw a binary
# clinical outcome (1=recovered, 0=not recovered), then computes the utility
# score which accounts for patient severity.
#
# KEY DESIGN DECISIONS:
# - clinical_outcome is used to UPDATE bandit statistics (arm selection logic)
# - utility_score is used to COMPUTE cumulative reward (performance metric)
# - Utility formula uses (1 - severity/10) so milder patients (severity=1)
#   benefit more (reward=0.9) than critical patients (severity=5, reward=0.5)

def simulate_outcome(medicine_index, severity):
    """
    Simulate a treatment outcome for a single patient.

    Parameters:
        medicine_index (int): Index of the medicine (0 to K-1)
        severity (int): Patient's disease severity score (1 to 5)

    Returns:
        clinical_outcome (int): 1 if recovered, 0 if not recovered
        utility_score (float): Reward value factoring in severity
            - Recovered + severity 1 -> 0.9 (mild case, high benefit)
            - Recovered + severity 5 -> 0.5 (critical case, lower benefit)
            - Not recovered -> 0.0 (no benefit regardless of severity)
    """
    # Draw recovery outcome using the medicine's hidden probability
    p_success = success_probabilities[medicine_index]
    clinical_outcome = np.random.binomial(1, p_success)

    # Compute utility: higher reward for milder cases when treatment succeeds
    utility_score = clinical_outcome * (1 - severity / 10)

    return clinical_outcome, utility_score


# ─── Quick verification of the helper function ───
print("Simulate Outcome — Verification:")
print(f"  {'Medicine':<10} {'Severity':<10} {'Outcome':<10} {'Utility':<10}")
print(f"  {'-'*40}")
for med, sev in [(0, 1), (0, 5), (4, 1), (4, 3), (4, 5)]:
    outcome, utility = simulate_outcome(med, sev)
    print(f"  Med {med:<6} Sev {sev:<6} {outcome:<10} {utility:<10.2f}")

print(f"\nUtility formula: outcome * (1 - severity/10)")
print(f"If outcome=1: sev=1 -> 0.9 | sev=5 -> 0.5 | outcome=0 -> 0.0")